In [1]:
import json
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
from rdkit import Chem, DataStructs
from rdkit.Chem import rdFingerprintGenerator

# Load model and data
model = joblib.load("../models/selected_model_rf.pkl")
rosemary = pd.read_csv("../data/cultivar_compound_catalogue.csv").drop_duplicates(
    "compound_id"
)
splits = json.loads(Path("../splits/split_indices.json").read_text())
curated = pd.read_csv("../data/curated_chembl.csv")

morgan_gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)


# 1. Generate features for Rosemary
def get_fp(smiles):
  mol = Chem.MolFromSmiles(smiles)
  return morgan_gen.GetFingerprint(mol) if mol else None


rosemary_fps = [get_fp(s) for s in rosemary["canonical_smiles"]]
X_rosemary = np.array([list(fp) for fp in rosemary_fps])

# 2. Predict ML scores
raw_probs = model.predict_proba(X_rosemary)[:, 1]

# Get validation raw probs for rank calculation
val_smiles = curated.iloc[splits["indices"]["validation"]]["canonical_smiles"]
X_val = np.array([list(get_fp(s)) for s in val_smiles])
val_probs = model.predict_proba(X_val)[:, 1]

ml_rank_scores = [(val_probs <= p).sum() / len(val_probs) for p in raw_probs]
rosemary["ML_raw_score"] = raw_probs
rosemary["ML_rank_score"] = ml_rank_scores

# 3. Applicability Domain (AD)
train_smiles = curated.iloc[splits["indices"]["train"]]["canonical_smiles"]
train_fps = [get_fp(s) for s in train_smiles]

# Calculate threshold
train_distances = []
for i, fp in enumerate(train_fps):
  sims = DataStructs.BulkTanimotoSimilarity(fp, train_fps)
  sims.pop(i)
  sims.sort(reverse=True)
  train_distances.append(np.mean([1 - s for s in sims[:5]]))
ad_threshold = np.percentile(train_distances, 95)

# Calculate Rosemary AD
ad_results = []
for fp in rosemary_fps:
  sims = DataStructs.BulkTanimotoSimilarity(fp, train_fps)
  sims.sort(reverse=True)
  dist = np.mean([1 - s for s in sims[:5]])
  ad_results.append(dist)

rosemary["ad_distance"] = ad_results
rosemary["ad_flag"] = np.where(
    rosemary["ad_distance"] <= ad_threshold, "within_AD", "extrapolation"
)

# Save
rosemary[[
    "compound_id",
    "ML_raw_score",
    "ML_rank_score",
    "ad_distance",
    "ad_flag",
]].to_csv("../results/ml_ad_scores.csv", index=False)
print("ML and AD scores saved.")

ML and AD scores saved.
